# Local smoke test

Plumbing only: tiny cache, tiny train, one predicted volume, one score. Not meant to produce a good model -- meant to prove the pipeline runs end to end before a real (GPU) run.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
from cell_tracking.config import get_train_dir, get_cache_dir
from cell_tracking.io_geff import list_geff_datasets
from cell_tracking.cache import build_volume_cache, cache_path

train_dir = get_train_dir()
cache_dir = get_cache_dir()
names = list_geff_datasets(train_dir)[:2]
print('smoke volumes:', names)
for name in names:
    build_volume_cache(train_dir / f'{name}.zarr', cache_path(cache_dir, name))
print('cache ready')

In [ ]:
from cell_tracking.train import History, train

history = History()
ckpt = train(
    train_dir,
    Path('dist/smoke/detector.pt'),
    epochs=2,
    frames_per_volume=4,
    val_frames_per_volume=2,
    names=names,
    cache_dir=cache_dir,
    history=history,
)
print('checkpoint:', ckpt)

In [ ]:
from cell_tracking.detect import load_model
from cell_tracking.predict import predict_volume, write_prediction

model, device = load_model(Path('dist/smoke/detector_best.pt'))
out_dir = Path('dist/smoke/preds')
out_dir.mkdir(parents=True, exist_ok=True)

for name in names:
    graph, stats = predict_volume(model, device, train_dir / f'{name}.zarr', cache_dir=cache_dir, t_max=20)
    write_prediction(graph, out_dir, name)
    print(name, stats)

In [ ]:
import subprocess
subprocess.run(
    ['python', '../scripts/score_local.py', '--geff-dir', str(out_dir), '--volume', *names],
    check=True,
)